# 14 · Market Basket Analysis — Apriori Recommendation Engine

**Added for CMPE 256 retail market-basket demonstration (PDF pages 7–11).** Independently testable and notebook-first.

This module recommends products frequently **co-purchased** in the supplied transactions using Apriori frequent itemsets and association rules. Support, confidence, and lift are computed from recorded basket frequencies. An exact cart match is required; for an unseen combination the recommender **abstains** instead of guessing.

> **Data honesty:** The bundled retail transactions are synthetic demonstrations, not observed transactions. These recommendations are unrelated to technical compatibility, clinical decisions, or Group 2's document-grounded RAG recommendations. The real `Retail.zip` was not supplied.

Run this notebook first for interactive exploration; the FastAPI bridge imports it automatically.

In [ ]:
%run 00_foundation.ipynb

In [ ]:
# Apriori association-rule market basket engine (offline; no LLM required)
from __future__ import annotations
from collections import Counter
from itertools import combinations
from pathlib import Path
import csv, json, uuid, math, sqlite3
from pydantic import BaseModel, Field, ConfigDict

BASKET_DATA_DIR=ROOT/'data'/'synthetic'

def init_basket_tables():
    with db_connect() as cx:
        cx.executescript("""
        CREATE TABLE IF NOT EXISTS basket_products (
            project_id TEXT NOT NULL,sku TEXT NOT NULL,name TEXT NOT NULL,category TEXT NOT NULL,
            PRIMARY KEY(project_id,sku));
        CREATE TABLE IF NOT EXISTS basket_transactions (
            project_id TEXT NOT NULL,transaction_id TEXT NOT NULL,source_type TEXT NOT NULL,
            created_at TEXT NOT NULL,PRIMARY KEY(project_id,transaction_id));
        CREATE TABLE IF NOT EXISTS basket_lines (
            project_id TEXT NOT NULL,transaction_id TEXT NOT NULL,sku TEXT NOT NULL,
            quantity INTEGER NOT NULL DEFAULT 1,
            PRIMARY KEY(project_id,transaction_id,sku),
            FOREIGN KEY(project_id,transaction_id) REFERENCES basket_transactions(project_id,transaction_id) ON DELETE CASCADE,
            FOREIGN KEY(project_id,sku) REFERENCES basket_products(project_id,sku) ON DELETE RESTRICT);
        CREATE INDEX IF NOT EXISTS basket_lines_by_sku ON basket_lines(project_id,sku);
        """)
    return True

class BasketRow(BaseModel):
    model_config=ConfigDict(extra='forbid')
    transaction_id:str=Field(min_length=1,max_length=100)
    sku:str=Field(min_length=1,max_length=100)
    product_name:str=Field(min_length=1,max_length=250)
    category:str=Field(default='Uncategorized',max_length=120)
    quantity:int=Field(default=1,ge=1,le=100000)

class BasketImportInput(BaseModel):
    model_config=ConfigDict(extra='forbid')
    rows:list[BasketRow]=Field(min_length=1,max_length=100_000)
    replace:bool=False

class BasketRecommendInput(BaseModel):
    model_config=ConfigDict(extra='forbid')
    cart:list[str]=Field(min_length=0,max_length=8)
    top_k:int=Field(default=5,ge=1,le=20)
    min_support:float=Field(default=0.05,ge=0.01,le=1.0)
    min_confidence:float=Field(default=0.4,ge=0.01,le=1.0)
    min_lift:float=Field(default=1.05,ge=1.0,le=50)
    max_itemset_size:int=Field(default=4,ge=2,le=5)

def import_basket_rows(project_id,rows,source_type='USER_DATASET',replace=False):
    """Project-scoped transactional batch ingest; cannot mix different transactions or projects."""
    pid=ensure_project_id(project_id)
    if not isinstance(rows,list) or not rows or len(rows)>100_000:raise ValueError('Provide between 1 and 100000 basket lines')
    if source_type not in ('SYNTHETIC_FIXTURE','USER_DATASET'):raise ValueError('Invalid source_type')
    validated=[BasketRow.model_validate(r) for r in rows]
    products={};groups={}
    for r in validated:
        if r.sku in products and products[r.sku]!=(r.product_name,r.category):
            raise ValueError(f'Conflicting product details for {r.sku}')
        products[r.sku]=(r.product_name,r.category)
        groups.setdefault(r.transaction_id,{})[r.sku]=groups.setdefault(r.transaction_id,{}).get(r.sku,0)+r.quantity
    if len(groups)<1:raise ValueError('No transactions found')
    with db_connect() as cx:
        if replace:
            cx.execute('DELETE FROM basket_lines WHERE project_id=?',(pid,))
            cx.execute('DELETE FROM basket_transactions WHERE project_id=?',(pid,))
            cx.execute('DELETE FROM basket_products WHERE project_id=?',(pid,))
        for sku,(name,category) in products.items():
            old=cx.execute('SELECT name,category FROM basket_products WHERE project_id=? AND sku=?',(pid,sku)).fetchone()
            if old and (old['name'],old['category'])!=(name,category):raise ValueError(f'Existing SKU metadata conflicts: {sku}')
            cx.execute('INSERT OR IGNORE INTO basket_products(project_id,sku,name,category) VALUES (?,?,?,?)',(pid,sku,name,category))
        for transaction_id,lines in groups.items():
            old=cx.execute('SELECT source_type FROM basket_transactions WHERE project_id=? AND transaction_id=?',(pid,transaction_id)).fetchone()
            if old:raise ValueError(f'Duplicate transaction ID: {transaction_id}. Set replace=True to replace project data.')
            cx.execute('INSERT INTO basket_transactions VALUES(?,?,?,?)',(pid,transaction_id,source_type,NOW()))
            cx.executemany('INSERT INTO basket_lines VALUES(?,?,?,?)',[(pid,transaction_id,sku,q) for sku,q in lines.items()])
    return {'project_id':pid,'transaction_count':len(groups),'product_count':len(products),'line_count':sum(map(len,groups.values())),
            'source_type':source_type,'is_synthetic':source_type=='SYNTHETIC_FIXTURE'}

def seed_market_basket_demo(project_id=PROJECT_ID,replace=True):
    p=BASKET_DATA_DIR/'market_basket_transactions_SYNTHETIC.csv'
    with p.open(newline='',encoding='utf-8') as f:rows=[{k:v for k,v in row.items() if k!='source_type'} for row in csv.DictReader(f)]
    return import_basket_rows(project_id,rows,source_type='SYNTHETIC_FIXTURE',replace=replace)

def basket_inventory(project_id):
    pid=ensure_project_id(project_id)
    with db_connect() as cx:
        products=[dict(r) for r in cx.execute('SELECT sku,name,category FROM basket_products WHERE project_id=? ORDER BY name',(pid,))]
        count=cx.execute('SELECT COUNT(*) FROM basket_transactions WHERE project_id=?',(pid,)).fetchone()[0]
        source_types=[r[0] for r in cx.execute('SELECT DISTINCT source_type FROM basket_transactions WHERE project_id=?',(pid,))]
    return {'project_id':pid,'transaction_count':count,'products':products,'source_types':source_types,'is_synthetic':bool(source_types) and all(s=='SYNTHETIC_FIXTURE' for s in source_types)}

def get_baskets(project_id):
    pid=ensure_project_id(project_id)
    with db_connect() as cx:
        rows=cx.execute('SELECT transaction_id,sku FROM basket_lines WHERE project_id=? ORDER BY transaction_id,sku',(pid,)).fetchall()
    grouped={}
    for r in rows:grouped.setdefault(r['transaction_id'],set()).add(r['sku'])
    return list(grouped.values())

def apriori_frequent_itemsets(baskets,min_support=0.05,max_itemset_size=4):
    """Apriori candidate generation with downward-closure pruning, based on transaction presence."""
    if not 0<min_support<=1:raise ValueError('min_support must be 0..1')
    if not 2<=max_itemset_size<=5:raise ValueError('max_itemset_size must be between 2 and 5')
    n=len(baskets)
    if not n:return {}
    minimum=math.ceil(min_support*n-1e-12)
    singleton=Counter(item for basket in baskets for item in basket)
    last={frozenset((sku,)):count for sku,count in singleton.items() if count>=minimum}
    counts=dict(last)
    for k in range(2,max_itemset_size+1):
        if not last:break
        prev=set(last)
        candidates=set()
        for a,b in combinations(prev,2):
            union=a|b
            if len(union)==k and all(frozenset(ss) in prev for ss in combinations(union,k-1)):
                candidates.add(union)
        if not candidates:break
        cur={candidate:sum(candidate.issubset(basket) for basket in baskets) for candidate in candidates}
        last={candidate:cnt for candidate,cnt in cur.items() if cnt>=minimum}
        counts.update(last)
    return counts

def association_rules(frequent_counts,n,min_confidence=0.4,min_lift=1.05):
    """Mine single-consequent rules X -> y with explainable support/confidence/lift."""
    if not n:return []
    rules=[]
    for itemset,intersection_count in frequent_counts.items():
        if len(itemset)<2:continue
        for item in itemset:
            antecedent=frozenset(itemset-{item})
            a_count=frequent_counts.get(antecedent,0)
            c_count=frequent_counts.get(frozenset([item]),0)
            if not a_count or not c_count:continue
            confidence=intersection_count/a_count
            support=intersection_count/n
            lift=confidence/(c_count/n)
            if confidence+1e-12<min_confidence or lift+1e-12<min_lift:continue
            rules.append({'antecedent':sorted(antecedent),'consequent':item,'support':round(support,6),
              'confidence':round(confidence,6),'lift':round(lift,6),
              'antecedent_transactions':a_count,'joint_transactions':intersection_count,'consequent_transactions':c_count})
    return sorted(rules,key=lambda r:(-r['confidence'],-r['lift'],-r['joint_transactions'],r['consequent']))

def market_basket_recommend(project_id,cart,top_k=5,min_support=0.05,min_confidence=0.4,
                            min_lift=1.05,max_itemset_size=4):
    pid=ensure_project_id(project_id)
    settings=BasketRecommendInput.model_validate(dict(cart=cart,top_k=top_k,min_support=min_support,
                                     min_confidence=min_confidence,min_lift=min_lift,max_itemset_size=max_itemset_size))
    if len(set(settings.cart))!=len(settings.cart):raise ValueError('Cart must contain unique product SKUs')
    inventory=basket_inventory(pid);catalog={p['sku']:p for p in inventory['products']}
    chosen=frozenset(settings.cart)
    base={'project_id':pid,'cart':settings.cart,'recommendations':[],'abstained':True,
          'training_transactions':inventory['transaction_count'],'source_types':inventory['source_types'],
          'is_synthetic':inventory['is_synthetic'],'algorithm':'Apriori association rules (exact-cart antecedent)',
          'parameters':settings.model_dump(exclude={'cart'}),'note':'Purchase co-occurrence does not establish hardware compatibility or product suitability.'}
    if not chosen:return {**base,'reason':'empty_cart'}
    if not chosen.issubset(catalog):return {**base,'reason':'unknown_sku','unknown_skus':sorted(chosen-set(catalog))}
    if len(chosen)>=settings.max_itemset_size:return {**base,'reason':'cart_too_large_for_mined_itemsets'}
    baskets=get_baskets(pid)
    if not baskets:return {**base,'reason':'no_transaction_history'}
    whole_cart_count=sum(chosen.issubset(b) for b in baskets)
    if not whole_cart_count:return {**base,'reason':'no_historical_cart_pattern'}
    frequent=apriori_frequent_itemsets(baskets,settings.min_support,settings.max_itemset_size)
    rules=association_rules(frequent,len(baskets),settings.min_confidence,settings.min_lift)
    filtered=[r for r in rules if frozenset(r['antecedent'])==chosen and r['consequent'] not in chosen]
    scored=[]
    for r in filtered:
        # Stable, interpretable score; strong confidence matters more than lift.
        score=0.6*r['confidence']+0.25*(min(r['lift'],3.0)/3.0)+0.15*r['support']
        scored.append({'sku':r['consequent'],'product_name':catalog[r['consequent']]['name'],
          'category':catalog[r['consequent']]['category'],'score':round(score,6),
          'confidence':r['confidence'],'lift':r['lift'],'support':r['support'],
          'evidence':{'antecedent_transactions':r['antecedent_transactions'],
                      'joint_transactions':r['joint_transactions'],
                      'total_transactions':len(baskets)},
          'reason':f"Co-purchased in {r['joint_transactions']} of {r['antecedent_transactions']} historical orders containing the full current cart."})
    scored.sort(key=lambda r:(-r['score'],-r['confidence'],r['sku']))
    if not scored:return {**base,'reason':'no_supported_association_rules','matching_cart_transactions':whole_cart_count}
    return {**base,'recommendations':scored[:settings.top_k],'abstained':False,'reason':None,
            'matching_cart_transactions':whole_cart_count,'mined_rule_count':len(rules)}

init_basket_tables()
print('Apriori basket recommender ready; call seed_market_basket_demo() to load synthetic transactions.')